# Run Validation inner-source module from Oxygen on AWS EMR

<a name="top"></a>

## Table Of Contents

  1. [Introduction](#intro)
  2. [Import libraries](#imports)
  3. [Prepare your environment](#prepare)
  4. [Prepare Local Files](#prepLocalFiles)
  5. [Build Validation jar](#buildjar)
  6. [Upload files to S3](#uploadtos3)  
    * [List files in S3](#lss3)  
    * [Delete a bucket in S3 (raw code)](#rms3)  
  
  7. [Start up EMR-servered on your AWS account](#startemr-servered)  
    * [Check status of EMR servered cluster](#emrserveredstatus)  
    * [Shut down an EMR-servered cluster](#termemrservered)  
  

<a name="intro"></a>

## Introduction

This notebook builds on the example in [laptop notebook](../laptop/runValidation_on_laptop.ipynb) to show how to **run Validation module on AWS EMR-serverless**.  The focus of this notebook is running Validation on a cluster at scale.  It's expected that you've already understood and developed the example using Validation example in the laptop notebook.  Those features are useable here as well, though that is not the focus of this notebook.


## Prepare to run validation  

In order to run validation you will need to build the validation jar using mvn.  To do this, run the following command:  

```bash
prompt> ./build_validation.sh
```

In [2]:
!cd ../;sh build_validation.sh

Setting up maven and java
/Users/c80774b/Documents/otp/java/java11
/Users/c80774b/opt/homebrew/Homebrew/Cellar/maven/3.9.0
/Users/c80774b/Documents/development/oxygen-validation/boto3/bin:/Users/c80774b/opt/homebrew/opt/openssl/bin:/Users/c80774b/opt/homebrew/Homebrew/bin:/Users/c80774b/opt/homebrew/Homebrew/bin:/Users/c80774b/opt/homebrew/Homebrew/sbin:/Library/Frameworks/Python.framework/Versions/3.9/bin:/Users/c80774b/opt/homebrew:/Users/c80774b/opt/homebrew/Homebrew/bin:/Users/c80774b/opt/homebrew/Homebrew/Cellar/maven/3.9.0/bin:/Users/c80774b/opt/homebrew/opt/openssl/bin:/Users/c80774b/.docker/bin:/Users/c80774b/opt/homebrew/Homebrew/sbin:/usr/local/bin:/usr/bin:/bin:/usr/sbin:/sbin:/Applications/CyberArk 
EPM.app/Contents/Helpers:/Users/c80774b/Library/Python/3.9/bin:/Users/c80774b/opt/homebrew/Homebrew/opt/python@3.11/libexec/bin:/Users/c80774b/Documents/otp/java/java11:/Users/c80774b/opt/homebrew/Homebrew/Cellar/maven/3.9.0:/Users/c80774b/opt/homebrew/Homebrew/Cellar/maven/3.9.

In [2]:
# Imports
# from tabulate import tabulate
import re
import sys
import os
import json
import pandas as pd
import datetime
import time
from importlib import reload
import boto3
import botocore.exceptions
import utils

# Messy but works to point to/import the utils module
root_jup = '/'.join(os.getcwd().split('/')[:-1])
# Add to search path
sys.path.append(root_jup)

In [6]:
print(json.__version__)

2.0.9


[Return to Top](#top)

<a name="prepare"></a>

## Prepare your environment to run AWS EMR
For more information, refer to the [Bit Bucket Validation on AWS setup page](https://code.experian.local/projects/EEO/repos/oxygen-validation/browse/AWS?at=develop)

1. Presumably you already have access to an Experian AWS account with VPC and S3 access setup.  If you don't have this already you will need to request access with the assistance of Experian Cloud Computing group.  You will need the secret key file to be able to login to your VPC using ssh.

2. Confirm you have access to the account by going through the AWS web console window:
  * Go to [https://experian.okta.com/app](https://experian.okta.com/app)
  * Click on AWS --> EEC AWS account _your account number here_

3. It's also assumed that you have already installed all of the pre-requisites detailed in the [laptop installation.](https://code.experian.local/projects/EEO/repos/oxygen-validation/browse/laptop?at=develop)

4.  Install AWS CLI from the Self Service App.

5. Configure your AWS CLI with your secret access key:  

    `aws configure`   
        `  Access Key ID: <Your Access Key here>  `  
        `  [****************OQXF]:   `  
        `  AWS Secret Access Key [****************zX9A]:`     
        `  Default region name [eu-west-1]:  `   
        `  Default output format  `  

        



[Return to Top](#top)

<a name="prepLocalFiles"></a>

## Prep Local files

Prior to uploading files to S3 and running EMR you need to prepare the following files with custom details related to your setup:  
  1. In your conf directory, in the file env.conf and validation-application.conf:
    A. Make sure to set the paths to the proper S3 file directory.
    

<a name="uploadtos3"></a>

## Upload files (jar, data, configs) to S3

In [22]:
# Setup s3 stuff

region = 'eu-west-1'
bucketname = 'esteban-test-emr'
upload = True

# Create a bucket in S3 (if it doesn't already exist.  Otherwise flag the exception)
s3 = boto3.client('s3', region_name = region)
location = {'LocationConstraint': region}
s3_root = 's3://' + bucketname + '/'
try:
    queue = s3.create_bucket(Bucket = bucketname, CreateBucketConfiguration=location)    
except botocore.exceptions.ClientError as error:
    print('Bucket is already there!')


# Method to upload files in a directory to S3
def dir2s3(localrootdir, s3root, filter = False, patlist = []):
    for root, dirs, files in os.walk(localrootdir):
        relroot = re.sub(localrootdir, '', root)
        for file in files:
            localfile = os.path.join(root,file)
            boolval = True
            if filter:
                boollist = [re.search(pat, localfile) for pat in patlist]
                boolval = all(boollist)
            if boolval:
                s3fname = os.path.join(relroot,file)
                print(s3root + s3fname)
                resp = s3.upload_file(localfile, bucketname, s3root + s3fname)
    print('\n\n')

def file2s3(localfile, s3root):
    fname = localfile.split('/')[-1]
    s3path = s3root + fname
    print(s3path)
    resp = s3.upload_file(localfile, bucketname, s3path)
    print('\n\n')

s3confroot = 'validation/conf/'
s3datarootdiscovery = 'validation/workspace/discovery/'
s3datarootdiscovered = 'validation/workspace/discovered/'
s3jarroot = 'validation/lib/'

if upload:
    # Upload conf files
    localrootdir = './config/'
    print('Uploading config files to S3.')
    dir2s3(localrootdir, s3confroot, filter = False)
    #localfile = './config/configs.tar.gz'
    #print('Uploading config file to S3.')
    #file2s3(localfile, s3confroot)
    print('Uploaded')
    
    # Upload example data files
    localrootdir = '../example/discovered/'
    print('Uploading input data files to S3.')
    dir2s3(localrootdir, s3datarootdiscovery, filter = False)
    print('Uploadeed')
    dir2s3(localrootdir, s3datarootdiscovered, filter = False)
    print('Uploaded')
    
    # Upload jars
    localrootdir = '../target'
    print('Uploading jar file to S3.')
    dir2s3(localrootdir, s3jarroot, filter = True, patlist = ['.jar'])
    # dir2s3(localrootdir, s3jarroot, filter = True, patlist = ['innersource-1.0', '.jar'])
    print('Uploaded')



Bucket is already there!
Uploading config files to S3.
validation/conf/env.conf
validation/conf/validation-application.conf



Uploaded
Uploading input data files to S3.
validation/workspace/discovery/E-230001.20161113.T



Uploadeed
validation/workspace/discovered/E-230001.20161113.T



Uploaded
Uploading jar file to S3.
validation/lib/oxygen-validation-innersource-1.0-jar-with-dependencies.jar
validation/lib/oxygen-validation-innersource-1.0.jar



Uploaded


<a name="lss3"></a>

## (optional) List Files on S3

In [18]:
# List the file paths matching multiple descriptor patterns (like ls | grep pat1 | grep pat ...)

filter = True   # Apply a reg exp to filter which files are listed
idfiles = False  # True => list files and put file names into a list,  False => Download the listed files into local paths

region = 'eu-west-1'

s3 = boto3.client('s3', region_name = region)

listbucket = bucketname
bucketexists = True

try:
    s3.head_bucket(Bucket = listbucket)
except:
    print('Bucket not found.')
    bucketexists = False
    pageitr = []
    


patlist = ['validation/lib', '.jar']
if bucketexists:
    paginator = s3.get_paginator('list_objects')
    pageitr = paginator.paginate(Bucket = listbucket)

stop = 0

itrnotempty = True
# lpitr = list(pageitr)
# if len(lpitr) == 1 and lpitr[0].get('Contents')  is None:
#     itrnotempty = False
#     print("Bucket is empty.")

def listS3Files(pageitr, filter, patlist, verbose = True):
    fnamelist = []
    for page in pageitr:
        for obj in page['Contents']:
            fname = obj['Key']
            date = obj['LastModified']
            owner = obj['Owner']['DisplayName']
            boollist = [True]
            if filter:
                boollist = [re.search(pat, fname) for pat in patlist]
            if all(boollist):
                fnamelist.append(fname)
                if verbose:
                    print(fname)
                    print(date)
                    print(owner)
                    print('\n\n')
    return fnamelist







if idfiles and itrnotempty:
    nflist = listS3Files(pageitr, filter, patlist)

    localdlpath = '../target/'
    # nflist = [f for f in fnamelist if re.search('steps', f)]
    # nflist = fnamelist
    
    # Optionally choose to download files from an S3 bucket from generated filename list
    download = not(idfiles)
    if download:
        s3b = boto3.resource('s3')
        for key in nflist:
            # Strip out fname
            fname = key.split('/')
            fname = '_'.join(fname)
            fullpath = localdlpath + fname
            s3b.Bucket(listbucket).download_file(key, fullpath)
else:
    print('Empty list')

Empty list


[Return to Top](#top)

<a name="rms3"></a>

## (optional) Remove files in S3
This is a raw code block which you can enable by changing to code for the case where you want to delete all files and delete an entire bucket.  User beware!

In [15]:
# patch-log4j-emr-5.29.0-v1.sh
# Delete objects in a bucket, then delete the bucket
# Use resource (higher-level abstraction)
s3res = boto3.resource('s3')
bucket = s3res.Bucket(listbucket)

bucket.objects.all().delete()
delbucket = True

# Now delete the bucket
if delbucket:
    response = s3.delete_bucket(Bucket = listbucket)
    print(response)

{'ResponseMetadata': {'RequestId': 'NSN351A3P9J6PP73', 'HostId': 'ziWBU9M1BZgZ6eOgMqwA2+qe0z0y+1YrIFowi4JFtdSwYAk7y0c6rFk1jpXbyLMjOjz0MdLqjRt72NPDdB+AUw==', 'HTTPStatusCode': 204, 'HTTPHeaders': {'x-amz-id-2': 'ziWBU9M1BZgZ6eOgMqwA2+qe0z0y+1YrIFowi4JFtdSwYAk7y0c6rFk1jpXbyLMjOjz0MdLqjRt72NPDdB+AUw==', 'x-amz-request-id': 'NSN351A3P9J6PP73', 'date': 'Tue, 22 Aug 2023 21:03:46 GMT', 'server': 'AmazonS3'}, 'RetryAttempts': 0}}


[Return to Top](#top)

<a name = "startemr-servered"></a>

## Start a servered EMR Cluster

In [19]:
# Parameters for EMR:
# Tags -- Modify these to use your own tags
tags = 'Rapid7Tag=Servers EITS UK - Cloud Agent UK - AWS 034279330187 - Sandbox'

# EC2 attributes -- Fill in keytab name as well as names of security groups and other account-specific information


ec2_attributes = '{"InstanceProfile":"BURoleForEMREC2Oxygen","EmrManagedMasterSecurityGroup":"sg-0099e4586ba8df0d6","EmrManagedSlaveSecurityGroup":"sg-0da93d286d4a5a5d0","KeyName":"ukidatafabric-sandbox-keypair-oxygen-01","AdditionalMasterSecurityGroups":["sg-08f5918b9833ca404"],"AdditionalSlaveSecurityGroups":["sg-02606dca78ec727d1","sg-08f5918b9833ca404"],"ServiceAccessSecurityGroup":"sg-084b35f61728fa0f4","SubnetId":"subnet-072f7eddde4ae9df8"}'
key_name = "ukidatafabric-sandbox-keypair-oxygen-01"
# subnetId = "subnet-00c4a9e6170220023"
subnetId = "subnet-072f7eddde4ae9df8" # Changed to subnet 3A eu-west-1c
AdditionalSlaveSecurityGroups = ["sg-02606dca78ec727d1","sg-08f5918b9833ca404"]
InstanceProfile = "BURoleForEMREC2Oxygen"
ServiceAccessSecurityGroup = "sg-084b35f61728fa0f4"
EmrManagedSlaveSecurityGroup = "sg-0da93d286d4a5a5d0"
EmrManagedMasterSecurityGroup = "sg-0099e4586ba8df0d6"
AdditionalMasterSecurityGroups = ["sg-08f5918b9833ca404"]
AdditionalSlaveSecurityGroups = ["sg-02606dca78ec727d1","sg-08f5918b9833ca404"]
log_uri = 's3://' + bucketname + '/emr/'

instance_groups = '[{"InstanceCount":1,"InstanceGroupType":"MASTER","Name":"Primary","InstanceType":"m5.xlarge","EbsConfiguration":{"EbsBlockDeviceConfigs":[{"VolumeSpecification":{"VolumeType":"gp2","SizeInGB":32},"VolumesPerInstance":2}]},"Configurations":[{"Classification":"hadoop-env","Configurations":[{"Classification":"export","Properties":{"JAVA_HOME":"/usr/lib/jvm/java-11-openjdk"}}],"Properties":{}},{"Classification":"spark-env","Configurations":[{"Classification":"export","Properties":{"JAVA_HOME":"/usr/lib/jvm/java-11-openjdk"}}],"Properties":{}},{"Classification":"hive-site","Properties":{"hive.metastore.client.factory.class":"com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory"}},{"Classification":"spark-defaults","Properties":{"spark.executorEnv.JAVA_HOME":"/usr/lib/jvm/java-11-openjdk"}}]},{"InstanceCount":2,"InstanceGroupType":"CORE","Name":"Core","InstanceType":"m5.xlarge","EbsConfiguration":{"EbsBlockDeviceConfigs":[{"VolumeSpecification":{"VolumeType":"gp2","SizeInGB":32},"VolumesPerInstance":2}]},"Configurations":[{"Classification":"hadoop-env","Configurations":[{"Classification":"export","Properties":{"JAVA_HOME":"/usr/lib/jvm/java-11-openjdk"}}],"Properties":{}},{"Classification":"spark-env","Configurations":[{"Classification":"export","Properties":{"JAVA_HOME":"/usr/lib/jvm/java-11-openjdk"}}],"Properties":{}},{"Classification":"hive-site","Properties":{"hive.metastore.client.factory.class":"com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory"}},{"Classification":"spark-defaults","Properties":{"spark.executorEnv.JAVA_HOME":"/usr/lib/jvm/java-11-openjdk"}}]}]'
configs = '[{"Classification":"hive-site","Properties":{"hive.metastore.client.factory.class":"com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory"}}]'
bootstrap_actions = '[]'
ebs_root_vol_size = 100
security_config_name = 'Oxygen Spark'
cluster_name= 'emr6.10-spark-innersource-validation'
# or
# cluster_name = 'ukidatafabric-sandbox-emr5.29-spark-ht-01'
# release_label = 'emr-5.29.0'
release_label = 'emr-6.10.0'
region = 'eu-west-1'
# datafile = 'part1-pin-hc.json'
print("Configuration is ready!")

Configuration is ready!


[Back to Top](#top)

### Generate string of jar and config files to be used in spark-submit

In [20]:
s3 = boto3.client('s3', region_name = region)
paginator = s3.get_paginator('list_objects')

# Determine config files to send to spark
pageitr = paginator.paginate(Bucket = listbucket)   
conffiles = [s3_root + f for f in listS3Files(pageitr, filter = True, patlist = [s3confroot], verbose = False)]
conffstring = ','.join(conffiles)

# Determine jar files to send to spark
pageitr = paginator.paginate(Bucket = listbucket)
jarfiles = [s3_root + f for f in listS3Files(pageitr, filter = True, patlist = [s3jarroot, 'jar'], verbose = False)]
jarstring = ','.join(jarfiles)

print(conffstring)
print()
print(jarstring)

s3://esteban-test-emr/validation/conf/env.conf,s3://esteban-test-emr/validation/conf/validation-application.conf

s3://esteban-test-emr/validation/lib/oxygen-validation-innersource-1.0-jar-with-dependencies.jar,s3://esteban-test-emr/validation/lib/oxygen-validation-innersource-1.0.jar


[Back to Top](#top)

### Start the servered EMR job (will shut-down after completing steps)

In [21]:
# Start servered EMR using boto3 using programmed paths / S3 files running spark command

job_name = 'boto3_esteban_emr'
s3_main_jar_path = 's3://' + bucketname + '/' + s3jarroot + 'oxygen-validation-innersource-1.0-jar-with-dependencies.jar'

paramlist = ['--verbose', '--master', "yarn",'--deploy-mode', "cluster",
             '--executor-cores', '4',
             '--executor-memory', "7G",
             '--num-executors', '34',
             '--driver-memory', "7G", 
             '--conf', "spark.dynamicAllocation.enabled=false",
             '--conf', "spark.yarn.submit.waitAppCompletion=true",
             '--jars', jarstring,
             '--files', conffstring,
             '--conf', "spark.dynamicAllocation.enabled=false",
             '--conf', "spark.driver.extraClassPath=/usr/lib/hadoop-lzo/lib/*:/usr/lib/hadoop/hadoop-aws.jar:/usr/share/aws/aws-java-sdk/*:/usr/share/aws/emr/emrfs/conf:/usr/share/aws/emr/emrfs/lib/*:/usr/share/aws/emr/emrfs/auxlib/*:/usr/share/aws/emr/goodies/lib/emr-spark-goodies.jar:/usr/share/aws/emr/security/conf:/usr/share/aws/emr/security/lib/*:/usr/share/aws/hmclient/lib/aws-glue-datacatalog-spark-client.jar:/usr/share/java/Hive-JSON-Serde/hive-openx-serde.jar:/usr/share/aws/sagemaker-spark-sdk/lib/sagemaker-spark-sdk.jar:/usr/share/aws/emr/s3select/lib/emr-s3-select-spark-connector.jar",
             '--conf', "spark.executor.extraClassPath=/usr/lib/hadoop-lzo/lib/*:/usr/lib/hadoop/hadoop-aws.jar:/usr/share/aws/aws-java-sdk/*:/usr/share/aws/emr/emrfs/conf:/usr/share/aws/emr/emrfs/lib/*:/usr/share/aws/emr/emrfs/auxlib/*:/usr/share/aws/emr/goodies/lib/emr-spark-goodies.jar:/usr/share/aws/emr/security/conf:/usr/share/aws/emr/security/lib/*:/usr/share/aws/hmclient/lib/aws-glue-datacatalog-spark-client.jar:/usr/share/java/Hive-JSON-Serde/hive-openx-serde.jar:/usr/share/aws/sagemaker-spark-sdk/lib/sagemaker-spark-sdk.jar:/usr/share/aws/emr/s3select/lib/emr-s3-select-spark-connector.jar",
             '--conf', "spark.executor.extraJavaOptions=-XX:+UseG1GC",
             '--conf', "spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version=2",
             '--conf', "spark.driver.extraJavaOptions=-Dfile.encoding=utf-8",
             '--conf', "spark.executor.extraJavaOptions=-Dfile.encoding=utf-8",
             '--class', 'com.experian.edf.oxygen.validation.app.SparkValidationApp',
             s3_main_jar_path,
             'env.conf',
             'validation-application.conf'
            ]

client = boto3.client('emr', region_name = region)
response = client.run_job_flow(Name = job_name,
                               LogUri = log_uri,
                               ReleaseLabel = release_label,
                               # CustomAmiId = '2.0.20230320.0',
                               # CustomAmiId = imageUri,
                               #CustomAmiId = 'ami-04f7efe62f419d9f5',
                               Instances = {
                                  'InstanceGroups': [
                                      {
                                          "Name":"Master - 1",
                                          "InstanceRole":"MASTER", 
                                          "InstanceType":"m5.xlarge",
                                          "InstanceCount":1,
                                          "EbsConfiguration":{
                                              "EbsBlockDeviceConfigs":[
                                                  {
                                                      "VolumeSpecification":{
                                                          'VolumeType':"gp2",
                                                          "SizeInGB":37
                                                      },
                                                      "VolumesPerInstance":2
                                                  }
                                              ]
                                            }
                                      },
                                      {
                                          "Name":"Core - 2",
                                          "InstanceRole":"CORE",
                                          "InstanceType":"m5.xlarge",
                                          "InstanceCount":2,
                                          "EbsConfiguration":{
                                              "EbsBlockDeviceConfigs":[
                                                  {
                                                     "VolumeSpecification":{
                                                         "SizeInGB":37,
                                                         "VolumeType":"gp2"
                                                     },
                                                      "VolumesPerInstance":2 
                                                  }
                                              ]
                                          }
                                      }
                                  ],
                                  'Ec2KeyName': key_name,
                                   'Ec2SubnetId': subnetId,
                                   'EmrManagedMasterSecurityGroup':EmrManagedMasterSecurityGroup,
                                   'EmrManagedSlaveSecurityGroup': EmrManagedSlaveSecurityGroup,
                                   'ServiceAccessSecurityGroup': ServiceAccessSecurityGroup,
                                   'AdditionalMasterSecurityGroups': AdditionalMasterSecurityGroups,
                                   'AdditionalSlaveSecurityGroups':  AdditionalSlaveSecurityGroups,  
                                   'KeepJobFlowAliveWhenNoSteps' : True # Set to False to terminate after finishing
                              },
                               Configurations = [
                                   {
                                    "Classification": "hadoop-env",
                                    "Configurations": [
                                          {
                                            "Classification": "export",
                                            "Properties": {
                                            "JAVA_HOME": "/usr/lib/jvm/java-11-openjdk"
                                            }
                                          }
                                    ],
                                "Properties": {}
                                  },
                                  {
                                    "Classification": "spark-env",
                                    "Configurations": [
                                      {
                                        "Classification": "export",
                                        "Properties": {
                                          "JAVA_HOME": "/usr/lib/jvm/java-11-openjdk"
                                        }
                                      }
                                    ],
                                    "Properties": {}
                                  },
                                {
                                        "Classification": "hive-site",
                                        "Properties": {
                                          "hive.metastore.client.factory.class": "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory"
                                        }
                                },
                                {
                                        "Classification": "spark-defaults",
                                        "Properties": {
                                          "spark.executorEnv.JAVA_HOME": "/usr/lib/jvm/java-11-openjdk"
                                        }
                                      }
                               ],
                                   # {
                                   #     "Classification": "spark-env",
                                   #     "Configurations": [
                                   #         {
                                   #             "Classification": "export",
                                   #             "Properties": {
                                   #                 "JAVA_HOME": "/usr/lib/jvm/java-17-amazon-corretto.x86_64"
                                   #             }
                                   #         }
                                   #     ]
                                   # },
                                   # {
                                   #     "Classification": "spark-defaults",
                                   #     "Properties": {
                                   #         "spark.executorEnv.JAVA_HOME": "/usr/lib/jvm/java-17-amazon-corretto.x86_64",
                                   #         "spark.driver.defaultJavaOptions": "-XX:OnOutOfMemoryError='kill -9 %p' -XX:MaxHeapFreeRatio=70",
                                   #         "spark.executor.defaultJavaOptions": "-verbose:gc -Xlog:gc*::time -XX:+PrintGCDetails -XX:+PrintGCDateStamps -XX:OnOutOfMemoryError='kill -9 %p' -XX:MaxHeapFreeRatio=70 -XX:+IgnoreUnrecognizedVMOptions"
                                   #             }
                                   #  }
                               # BootstrapActions=[
                               #     {   
                               #         'Name': 'Custom action',
                               #         'ScriptBootstrapAction': {
                               #             'Path': 's3://oxygen-sandbox-s3-config-01/patch-log4j-emr-5.29.0-v1.sh',
                               #         }
                               #     },
                               # ],
                               EbsRootVolumeSize = ebs_root_vol_size,
                               SecurityConfiguration = security_config_name,    
                               VisibleToAllUsers = True,
                               ServiceRole = 'arn:aws:iam::034279330187:role/eec-aws-standard-emr-defaultrole',
                               Applications = [
                                   {
                                       'Name':'SPARK'
                                   },
                                   {
                                       'Name': 'HIVE'
                                   },
                                   {
                                       'Name' : 'HADOOP'
                                   }
                               ],
                               Tags = [
                                   {
                                       'Key':'Rapid7Tag',
                                       'Value':'Servers EITS UK - Cloud Agent UK - AWS 034279330187 - Sandbox'
                                   }
                               ],
                               
                               Steps = [
                                   {
                                            'Name': 'Run validation using command-runner.jar',
                                            # 'ActionOnFailure': 'TERMINATE_CLUSTER', # Set to TERMINATE_CLUSTER to terminate on error
                                       'ActionOnFailure': 'CONTINUE',     
                                       'HadoopJarStep': {
                                                'Jar': 'command-runner.jar',
                                                'Args': ["spark-submit", *paramlist
                                                    ]
                                            }
                                   }
                                ],
                               
                               JobFlowRole = 'BURoleForEMREC2Oxygen'
                               # JobFlowRole = 'arn:aws:iam::034279330187:role/BURoleFor-openshift-installer'
                                          
)
print(json.dumps(response, indent = 4, sort_keys = True, default = str))

{
    "ClusterArn": "arn:aws:elasticmapreduce:eu-west-1:034279330187:cluster/j-36FWOHQZGDRNF",
    "JobFlowId": "j-36FWOHQZGDRNF",
    "ResponseMetadata": {
        "HTTPHeaders": {
            "content-length": "118",
            "content-type": "application/x-amz-json-1.1",
            "date": "Tue, 22 Aug 2023 21:07:10 GMT",
            "x-amzn-requestid": "e6487b33-a2b6-4ae8-aae7-4ae415162a8a"
        },
        "HTTPStatusCode": 200,
        "RequestId": "e6487b33-a2b6-4ae8-aae7-4ae415162a8a",
        "RetryAttempts": 0
    }
}


[Return to Top](#top)

<a name="emrserveredstatus"></a>

## List status of EMR servered clusters

In [25]:
# Find cluster jobID's by particular state (STARTING, BOOTSTRAPPING, WAITING, TERMINATING)
# j-3BRMWTZU9UOHX
response = client.list_clusters(
  ClusterStates = [
      'STARTING', 'BOOTSTRAPPING', 'WAITING', 'RUNNING','TERMINATING'
  ]
)
print(json.dumps(response, indent = 4, sort_keys = True, default = str))

{
    "Clusters": [
        {
            "ClusterArn": "arn:aws:elasticmapreduce:eu-west-1:034279330187:cluster/j-2PAM5283XIH4B",
            "Id": "j-2PAM5283XIH4B",
            "Name": "boto3_esteban_emr",
            "NormalizedInstanceHours": 0,
            "Status": {
                "State": "STARTING",
                "StateChangeReason": {},
                "Timeline": {
                    "CreationDateTime": "2023-08-21 11:47:07.736000-06:00"
                }
            }
        },
        {
            "ClusterArn": "arn:aws:elasticmapreduce:eu-west-1:034279330187:cluster/j-1CGVZDOXPXFQ7",
            "Id": "j-1CGVZDOXPXFQ7",
            "Name": "qa-sandbox-emr-java-11-sparkht-05",
            "NormalizedInstanceHours": 0,
            "Status": {
                "State": "WAITING",
                "StateChangeReason": {
                    "Message": "Cluster ready to run steps."
                },
                "Timeline": {
                    "CreationDateTime": "20

[Return to Top](#top)

<a name="termemrservered"></a>

## Shutdown an EMR Servered Cluster
This is raw code which can be enabled and run by changing the code cell to "code".  
  **User Beware!!**

In [153]:
# Terminate the EMR instance based on its ID
response = client.list_clusters(
    CreatedAfter=datetime.datetime(2023,3,17),
    ClusterStates = [
      'WAITING'
  ]
)
cluster_id = response['Clusters'][0]['Id']
print(cluster_id)

cluster_id = 'j-1VN33KU0R2PNT'

shutdown = True
if shutdown:
    response = client.terminate_job_flows(
        JobFlowIds = [cluster_id]
    )
    print(response)

j-1VN33KU0R2PNT
{'ResponseMetadata': {'RequestId': '39f6c2bc-07a2-400d-b684-4f68cf64314c', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '39f6c2bc-07a2-400d-b684-4f68cf64314c', 'content-type': 'application/x-amz-json-1.1', 'content-length': '0', 'date': 'Thu, 01 Jun 2023 23:52:16 GMT'}, 'RetryAttempts': 0}}


[Return to Top](#top)

## Print Status of job run

In [119]:
# while True:
jrunstatus_response = emr_sl.list_job_runs(
    applicationId=applicationId,
    # nextToken='string',
    # maxResults=123,
    # createdAtAfter=datetime(2015, 1, 1),
    # createdAtBefore=datetime(2015, 1, 1),
    # states=[
    #     'SUBMITTED'|'PENDING'|'SCHEDULED'|'RUNNING'|'SUCCESS'|'FAILED'|'CANCELLING'|'CANCELLED',
    # ]
    states=[
        'SUBMITTED', 'PENDING', 'SCHEDULED', 'RUNNING', 'SUCCESS'
    ]
)

print(json.dumps(jrunstatus_response, indent = 4, sort_keys = True, default = str))

{
    "ResponseMetadata": {
        "HTTPHeaders": {
            "connection": "keep-alive",
            "content-length": "957",
            "content-type": "application/json",
            "date": "Fri, 09 Jun 2023 21:01:54 GMT",
            "x-amz-apigw-id": "GRPydEipjoEF4CQ=",
            "x-amzn-remapped-content-length": "957",
            "x-amzn-remapped-date": "Fri, 09 Jun 2023 21:01:54 GMT",
            "x-amzn-remapped-x-amzn-requestid": "GRPydEipjoEF4CQ=",
            "x-amzn-requestid": "a4854713-b695-46bd-a91e-c1dbdbede242",
            "x-amzn-trace-id": "Root=1-64839342-30eb49a94621347c7cf90530"
        },
        "HTTPStatusCode": 200,
        "RequestId": "a4854713-b695-46bd-a91e-c1dbdbede242",
        "RetryAttempts": 0
    },
    "jobRuns": [
        {
            "applicationId": "00fanrpvceu8m00p",
            "arn": "arn:aws:emr-serverless:eu-west-1:034279330187:/applications/00fanrpvceu8m00p/jobruns/00fap6ke09s3ig0q",
            "createdAt": "2023-06-09 11:03:30.

[Return to top](#top)

<a name="termemr"></a>

## Shut down a serverless job

In [ ]:
response = emr_sl.cancel_job_run(
    applicationId = applicationId,
    jobRunId='00fanrqsg79rj00q'
)

## Shut down a serverless application

In [120]:
# Stop / Terminate EMR Serverless application

try:
    responsestop = emr_sl.stop_application(applicationId = applicationId)
    print(json.dumps(responsestop, indent = 4, sort_keys = True, default = str))
except:
    print('Application is already stopped or not available.')

try:
    responseappkill = emr_sl.delete_application(applicationId = applicationId)
    print(json.dumps(responseappkill, indent = 4, sort_keys = True, default = str))
    
except:
    print('Application either is already terminated or else somehow needs to be stopped first.')


{
    "ResponseMetadata": {
        "HTTPHeaders": {
            "connection": "keep-alive",
            "content-length": "2",
            "content-type": "application/json",
            "date": "Fri, 09 Jun 2023 21:07:26 GMT",
            "x-amz-apigw-id": "GRQmXEQcjoEFt4w=",
            "x-amzn-remapped-content-length": "2",
            "x-amzn-remapped-date": "Fri, 09 Jun 2023 21:07:26 GMT",
            "x-amzn-remapped-x-amzn-requestid": "GRQmXEQcjoEFt4w=",
            "x-amzn-requestid": "b49f2e34-fe9b-4e4d-a9b9-a4a5eae3c62e",
            "x-amzn-trace-id": "Root=1-6483948e-7c31d6517b5a99b309aab050"
        },
        "HTTPStatusCode": 200,
        "RequestId": "b49f2e34-fe9b-4e4d-a9b9-a4a5eae3c62e",
        "RetryAttempts": 0
    }
}
{
    "ResponseMetadata": {
        "HTTPHeaders": {
            "connection": "keep-alive",
            "content-length": "2",
            "content-type": "application/json",
            "date": "Fri, 09 Jun 2023 21:07:27 GMT",
            "x-amz-a

[Return to top](#top)